<a href="https://colab.research.google.com/github/kenichiy19/1CCPK-MLAM/blob/main/Global%20Solutions/2026_2/notebooks/01_preparacao_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Preparação dos dados
**Global Solution – MLAM | Desafio CitizenTech**

**Problema:** identificar fatores socioeconômicos associados à vulnerabilidade social nos municípios brasileiros, para orientar onde priorizar uma solução de apoio ao cidadão.

**Fonte:** PNUD; Ipea; FJP. *Atlas do Desenvolvimento Humano no Brasil* – Censo Demográfico (1991, 2000 e 2010). Disponível em: https://www.atlasbrasil.org.br/acervo/biblioteca. Acesso em: 9 nov. 2026.



**Integrantes:**
- Kenichi Caio Yamamoto – RM 569815
- Rodger Costa Rios – RM 571438

## 1. Preparação do ambiente
Importação das bibliotecas e envio da planilha original para o Colab.

In [1]:
import os
import pandas as pd

In [2]:
base_dados = 'censo_total_1991_2010.xlsx'

if not os.path.exists(base_dados):
    from google.colab import files
    files.upload()

print('Arquivo encontrado:', os.path.exists(base_dados))

Arquivo encontrado: True


## 2. Leitura e filtro dos dados
A planilha original é lida sem alterações. Em seguida, são mantidos apenas os registros de 2010, o período de referência mais recente disponível por município nessa base.

In [3]:
dados_brutos = pd.read_excel(base_dados, sheet_name='MUN 91-00-10')

siglas = pd.read_excel(base_dados, sheet_name='Siglas')

print('Dimensões da base original (linhas, colunas):', dados_brutos.shape)
print('\nRegistros por ano:')
for ano, total in dados_brutos['ANO'].value_counts().sort_index().items():
    print(f'  {ano}: {total} municípios')

Dimensões da base original (linhas, colunas): (16695, 237)

Registros por ano:
  1991: 5565 municípios
  2000: 5565 municípios
  2010: 5565 municípios


In [4]:
dados_2010 = dados_brutos[dados_brutos['ANO'] == 2010].copy()
print('Municípios em 2010:', len(dados_2010))

Municípios em 2010: 5565


## 3. Seleção e verificação das variáveis

| Sigla | Descrição | Papel na análise |
|---|---|---|
| PPOB | % da população vulnerável à pobreza | Variável principal |
| T_ANALF18M | Taxa de analfabetismo – 18 anos ou mais (%) | Explicativa |
| P_FORMAL | Grau de formalização do trabalho (%) | Explicativa |
| T_BANAGUA | % da população em domicílios com banheiro e água encanada | Explicativa |
| T_MED18M | % de adultos com ensino médio completo | Apoio |
| T_DES18M | Taxa de desocupação – 18 anos ou mais (%) | Apoio |

A renda per capita (RDPC) não foi usada como variável explicativa, pois o PPOB é calculado a partir da própria renda.

In [5]:
colunas_id = ['UF', 'Codmun7', 'Município']
colunas_analise = ['PPOB', 'T_ANALF18M', 'P_FORMAL', 'T_BANAGUA', 'T_MED18M', 'T_DES18M']

base = dados_2010[colunas_id + colunas_analise].reset_index(drop=True)

dicionario = siglas[siglas['SIGLA'].isin(colunas_analise)][['SIGLA', 'NOME LONGO']]
dicionario

,SIGLA,NOME LONGO
18,T_ANALF18M,Taxa de analfabetismo da população de 18 anos ...
66,T_MED18M,Percentual da população de 18 anos ou mais com...
80,PPOB,Proporção de vulneráveis à pobreza
109,P_FORMAL,Grau de formalização do trabalho das pessoas o...
132,T_DES18M,Taxa de desocupação da população de 18 anos ou...
139,T_BANAGUA,Percentual da população que vive em domicílios...


In [6]:
base.head()

,UF,Codmun7,Município,PPOB,T_ANALF18M,P_FORMAL,T_BANAGUA,T_MED18M,T_DES18M
0,11,1100015,ALTA FLORESTA D'OESTE,47.12,13.00,41.68,80.20,23.19,5.17
1,11,1100023,ARIQUEMES,29.04,8.53,52.81,85.35,34.58,4.57
2,11,1100031,CABIXI,46.98,14.73,39.38,91.63,23.89,2.58
3,11,1100049,CACOAL,29.82,8.94,55.07,93.44,31.03,5.61
4,11,1100056,CEREJEIRAS,34.85,11.17,45.54,86.87,27.79,5.42


In [7]:
print('Valores ausentes por coluna:')
for coluna, total in base.isna().sum().items():
    print(f'  {coluna}: {total}')
print('\nMunicípios duplicados:', base['Codmun7'].duplicated().sum())

Valores ausentes por coluna:
  UF: 0
  Codmun7: 0
  Município: 0
  PPOB: 0
  T_ANALF18M: 0
  P_FORMAL: 0
  T_BANAGUA: 0
  T_MED18M: 0
  T_DES18M: 0

Municípios duplicados: 0


## 4. Criação da variável PRIORITARIO
O município é considerado **prioritário (1)** quando 50% ou mais da população é vulnerável à pobreza (PPOB ≥ 50); caso contrário, recebe **0**. O limite foi escolhido por ser absoluto e de fácil interpretação.

In [8]:
base['PRIORITARIO'] = (base['PPOB'] >= 50).astype(int)

contagem = base['PRIORITARIO'].value_counts().sort_index()
proporcao = base['PRIORITARIO'].value_counts(normalize=True).sort_index() * 100
pd.DataFrame({'Municípios': contagem, 'Percentual (%)': proporcao.round(2)})

,Municípios,Percentual (%)
PRIORITARIO,,
0,3150,56.6
1,2415,43.4


## 5. Resumo estatístico e exportação
Resumo das variáveis selecionadas e gravação da base limpa em .csv, que será usada no notebook de análises.

In [9]:
base[colunas_analise].describe().round(2)

,PPOB,T_ANALF18M,P_FORMAL,T_BANAGUA,T_MED18M,T_DES18M
count,5565.00,5565.00,5565.00,5565.00,5565.00,5565.00
mean,43.99,17.40,43.51,80.87,24.75,6.19
std,22.44,10.70,19.27,21.71,8.68,3.66
min,1.97,0.97,2.97,3.26,3.04,0.00
25%,23.96,8.59,26.32,67.78,18.32,3.70
50%,42.23,14.11,42.85,91.25,23.48,5.66
75%,65.30,26.26,59.77,98.00,30.01,7.99
max,91.57,47.64,89.11,100.00,66.23,38.45


In [10]:
base.to_csv('base_municipios_2010.csv', index=False, encoding='utf-8-sig')
print('Base limpa salva. Dimensões finais:', base.shape)

from google.colab import files
files.download('base_municipios_2010.csv')

Base limpa salva. Dimensões finais: (5565, 10)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>